# Experiment Results Insights EDA

This notebook is the result-side companion to the source-data EDA notebooks. Instead of asking what the data looks like before modeling, it asks what the experiment outputs tell us about model behavior.

The first pass focuses on:

- model-build performance and robustness across periods
- feature transform effects, especially for regularized linear models
- feature-policy comparisons
- period-specific difficulty and COVID-era sensitivity
- large forecast errors and residual patterns worth investigating before promoting dashboard insights

The dashboard should eventually surface only the clearest, most durable takeaways from this notebook.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

ARTIFACT_DIR = Path("dashboard/public_artifacts/latest")
SCORE_COL = "selection_score_balanced"

print(f"Artifact directory: {ARTIFACT_DIR.resolve()}")


In [ ]:
def read_parquet(name: str) -> pd.DataFrame:
    path = ARTIFACT_DIR / f"{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_parquet(path)

leaderboard = read_parquet("model_leaderboard")
forecast_paths = read_parquet("forecast_paths")
performance = read_parquet("performance_over_time")
feature_family_summary = read_parquet("feature_family_summary")

manifest_path = ARTIFACT_DIR / "experiment_manifest.json"
champion_path = ARTIFACT_DIR / "champion_selection.json"
experiment_manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
champion = json.loads(champion_path.read_text()) if champion_path.exists() else {}

for frame_name, frame in {
    "leaderboard": leaderboard,
    "forecast_paths": forecast_paths,
    "performance": performance,
    "feature_family_summary": feature_family_summary,
}.items():
    print(f"{frame_name:24s} {frame.shape[0]:>10,} rows x {frame.shape[1]:>3,} columns")

print("Champion config:", champion.get("config_id") or champion.get("model_config_id"))
print("Experiment id:", experiment_manifest.get("experiment_id") or experiment_manifest.get("run_id"))


In [ ]:
FEATURE_TRANSFORM_LABELS = {
    "identity": "No transform",
    "log_signed": "Signed log",
    "quadratic": "Quadratic",
    "cubic": "Cubic",
    "log_signed_quadratic_cubic": "Signed log + quadratic + cubic",
}
FEATURE_TRANSFORM_ORDER = [
    "identity",
    "log_signed",
    "quadratic",
    "cubic",
    "log_signed_quadratic_cubic",
]

BASELINE_MODEL_FAMILIES = {"baseline"}
BASELINE_MODEL_BUILDS = {"seasonal_naive", "naive"}

MODEL_BUILD_LABELS = {
    "seasonal_naive": "Seasonal naive",
    "ridge": "Ridge",
    "lasso": "Lasso",
    "elastic_net": "Elastic net",
    "arima": "ARIMA",
    "sarima": "SARIMA",
    "sarimax": "SARIMAX",
    "random_forest": "Random forest",
    "extra_trees": "Extra trees",
    "xgboost": "XGBoost",
    "gru": "GRU",
    "lstm": "LSTM",
}


def prepare_model_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "model_config_id" not in out and "config_id" in out:
        out["model_config_id"] = out["config_id"]
    if "feature_transform" not in out:
        out["feature_transform"] = "identity"
    out["feature_transform"] = out["feature_transform"].fillna("identity")
    out["feature_transform_label"] = out["feature_transform"].map(
        lambda value: FEATURE_TRANSFORM_LABELS.get(str(value), str(value).replace("_", " ").title())
    )
    if "model_build" in out:
        out["model_build_label"] = out["model_build"].map(
            lambda value: MODEL_BUILD_LABELS.get(str(value), str(value).replace("_", " ").title())
        )
    return out

leaderboard = prepare_model_frame(leaderboard)
forecast_paths = prepare_model_frame(forecast_paths)
performance = prepare_model_frame(performance)

candidate_leaderboard = leaderboard[
    ~leaderboard.get("model_family", "").isin(BASELINE_MODEL_FAMILIES)
    & ~leaderboard.get("model_build", "").isin(BASELINE_MODEL_BUILDS)
].copy()

if SCORE_COL not in candidate_leaderboard and {"mae", "rmse"}.issubset(candidate_leaderboard.columns):
    candidate_leaderboard[SCORE_COL] = 0.75 * candidate_leaderboard["mae"] + 0.25 * candidate_leaderboard["rmse"]

print("Candidate model configs:", len(candidate_leaderboard))
print("Model builds:", sorted(candidate_leaderboard["model_build_label"].dropna().unique()))
print("Feature transforms:", sorted(candidate_leaderboard["feature_transform_label"].dropna().unique()))


## 1. Top Configurations

Start with the simple leaderboard, but treat it as a doorway rather than the answer. A single global score can hide models that are strong in ordinary periods but fragile during disruption, or models that look good during recovery because they overfit the recovery regime.


In [ ]:
leaderboard_cols = [
    "config_id",
    "model_family",
    "model_build_label",
    "mode",
    "feature_family_name",
    "feature_policy",
    "feature_transform_label",
    SCORE_COL,
    "mae",
    "rmse",
    "r2",
    "diracc",
    "pre_covid_mae",
    "covid_shock_mae",
    "recovery_mae",
    "recent_mae",
]
leaderboard_cols = [col for col in leaderboard_cols if col in candidate_leaderboard]

top_configs = candidate_leaderboard.sort_values(SCORE_COL).loc[:, leaderboard_cols].head(25)
display(top_configs)


## 2. Model Build Summary

This view compresses the model grid to one row per model build. It helps separate questions like "did XGBoost have a strong candidate?" from "did the average XGBoost configuration perform well?" Both matter: the best row shows potential, while the median row shows how easy the family is to tune in this setup.


In [ ]:
build_summary = (
    candidate_leaderboard.groupby(["model_family", "model_build", "model_build_label"], dropna=False)
    .agg(
        configurations=("config_id", "nunique"),
        best_balanced_score=(SCORE_COL, "min"),
        median_balanced_score=(SCORE_COL, "median"),
        best_mae=("mae", "min"),
        median_mae=("mae", "median"),
        best_rmse=("rmse", "min"),
        best_r2=("r2", "max"),
    )
    .reset_index()
    .sort_values("best_balanced_score")
)

display(build_summary)

fig, ax = plt.subplots(figsize=(10, max(4, 0.42 * len(build_summary))))
plot_df = build_summary.sort_values("best_balanced_score", ascending=True)
ax.barh(plot_df["model_build_label"], plot_df["best_balanced_score"], color="#007f68")
ax.invert_yaxis()
ax.set_title("Best Balanced Score By Model Build")
ax.set_xlabel("Best balanced score (lower is better)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 3. Linear Transform Screening

This section looks specifically at regularized linear models because the nonlinear transform experiment was designed for that family. No transform is the untransformed baseline. Non-identity transform families keep the original selected features and append transformed terms, so a signed-log model includes both `x` and `sign(x) * log1p(abs(x))` terms.

The current transform grid applies a transform family broadly to selected numeric features, then relies on scaling and regularization to shrink weak terms. That is useful as a broad screen, but it is not the same as a variable-specific recipe like "log gas prices, quadratic service hours, linear CPI."


In [ ]:
linear_models = candidate_leaderboard[candidate_leaderboard["model_family"].eq("linear")].copy()

transform_summary = (
    linear_models.groupby(["model_build_label", "feature_transform", "feature_transform_label"], dropna=False)
    .agg(
        configurations=("config_id", "nunique"),
        best_balanced_score=(SCORE_COL, "min"),
        median_balanced_score=(SCORE_COL, "median"),
        best_mae=("mae", "min"),
        median_mae=("mae", "median"),
        best_rmse=("rmse", "min"),
        best_r2=("r2", "max"),
    )
    .reset_index()
)
transform_summary["transform_order"] = transform_summary["feature_transform"].map(
    lambda value: FEATURE_TRANSFORM_ORDER.index(value) if value in FEATURE_TRANSFORM_ORDER else len(FEATURE_TRANSFORM_ORDER)
)
transform_summary = transform_summary.sort_values(
    ["model_build_label", "transform_order", "best_balanced_score"]
).drop(columns=["transform_order"])

lasso_transforms = set(
    linear_models.loc[linear_models["model_build"].eq("lasso"), "feature_transform"].dropna().astype(str)
)
if lasso_transforms == {"identity"}:
    print(
        "Note: lasso appears with No transform only in this artifact bundle. "
        "The transform code supports lasso, but transformed lasso configs were not added/merged here."
    )

display(transform_summary.drop(columns=["feature_transform"]))

if not transform_summary.empty:
    pivot = transform_summary.pivot_table(
        index="feature_transform_label",
        columns="model_build_label",
        values="best_balanced_score",
        aggfunc="min",
    ).reindex([FEATURE_TRANSFORM_LABELS[key] for key in FEATURE_TRANSFORM_ORDER])
    display(pivot)
    ax = pivot.plot(kind="bar", figsize=(11, 5), rot=35)
    ax.set_title("Best Linear-Model Score By Transform Family")
    ax.set_ylabel("Best balanced score (lower is better)")
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


## 4. Feature Policy Screening

Feature policies answer a different question than feature families: after a candidate family exists, should the model use everything, prune correlated inputs, select variables by mutual information, or use a tree-based importance screen? This is a good place to look for whether input control matters as much as the feature idea itself.


In [ ]:
policy_summary = (
    candidate_leaderboard.groupby(["model_family", "model_build_label", "feature_policy"], dropna=False)
    .agg(
        configurations=("config_id", "nunique"),
        best_balanced_score=(SCORE_COL, "min"),
        median_balanced_score=(SCORE_COL, "median"),
        best_mae=("mae", "min"),
        best_rmse=("rmse", "min"),
        best_r2=("r2", "max"),
    )
    .reset_index()
    .sort_values(["model_family", "model_build_label", "best_balanced_score"])
)

display(policy_summary)


## 5. Feature Family Screening

Feature-family summaries should be read alongside model build and feature policy. A family can look strong because it contains useful signals, because a specific model can exploit it, or because a pruning policy removed the noisy parts. This section starts by ranking families by their best observed result.


In [ ]:
family_cols = [
    "feature_family_name",
    "mode",
    "best_selection_score_balanced",
    "best_rmse",
    "best_mae",
    "best_r2",
    "best_diracc",
    "avg_rmse",
    "avg_mae",
]
family_cols = [col for col in family_cols if col in feature_family_summary]
family_rank = feature_family_summary.sort_values(
    "best_selection_score_balanced" if "best_selection_score_balanced" in feature_family_summary else "best_selection_score"
).loc[:, family_cols]

display(family_rank.head(30))


## 6. Period Difficulty And Robustness

The project has a structural-break problem by design. Strong result analysis should therefore ask whether a model is merely good on average or whether it survives the pre-COVID, shock, recovery, and recent regimes without falling apart.


In [ ]:
period_specs = [
    ("Pre-COVID", "pre_covid_mae", "pre_covid_rmse"),
    ("COVID shock", "covid_shock_mae", "covid_shock_rmse"),
    ("Recovery", "recovery_mae", "recovery_rmse"),
    ("Recent", "recent_mae", "recent_rmse"),
]
period_rows = []
for label, mae_col, rmse_col in period_specs:
    if mae_col in candidate_leaderboard:
        best_mae = candidate_leaderboard[mae_col].min()
        median_mae = candidate_leaderboard[mae_col].median()
        period_rows.append(
            {
                "period": label,
                "best_mae": best_mae,
                "median_mae": median_mae,
                "best_vs_median_improvement": (median_mae - best_mae) / median_mae if median_mae else np.nan,
                "best_rmse": candidate_leaderboard[rmse_col].min() if rmse_col in candidate_leaderboard else np.nan,
                "median_rmse": candidate_leaderboard[rmse_col].median() if rmse_col in candidate_leaderboard else np.nan,
            }
        )
period_summary = pd.DataFrame(period_rows)
display(period_summary)

if not period_summary.empty:
    ax = period_summary.plot(x="period", y=["best_mae", "median_mae"], kind="bar", figsize=(10, 4), rot=0)
    ax.set_title("Period Difficulty: Best vs Median MAE")
    ax.set_ylabel("MAE")
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
period_mae_cols = [col for _, col, _ in period_specs if col in candidate_leaderboard]
robustness = candidate_leaderboard.copy()
if period_mae_cols:
    robustness["worst_period_mae"] = robustness[period_mae_cols].max(axis=1)
    robustness["period_mae_spread"] = robustness[period_mae_cols].max(axis=1) - robustness[period_mae_cols].min(axis=1)
    robustness["period_mae_mean"] = robustness[period_mae_cols].mean(axis=1)
    robustness["robustness_rank_score"] = 0.50 * robustness[SCORE_COL] + 0.30 * robustness["worst_period_mae"] + 0.20 * robustness["period_mae_spread"]

    robust_cols = [
        "config_id",
        "model_build_label",
        "mode",
        "feature_family_name",
        "feature_policy",
        "feature_transform_label",
        SCORE_COL,
        "worst_period_mae",
        "period_mae_spread",
        "period_mae_mean",
        *period_mae_cols,
    ]
    display(robustness.sort_values("robustness_rank_score").loc[:, robust_cols].head(25))


## 7. Large Error Inspection

Large errors are often more revealing than average errors. This section starts with the current best global model, then lists the largest misses and plots residuals over time. Useful follow-ups include comparing whether those same months are hard for many model families or only for the selected champion.


In [ ]:
best_config = candidate_leaderboard.sort_values(SCORE_COL).iloc[0]
best_config_id = best_config["config_id"]
print("Best config for large-error inspection:", best_config_id)
print(best_config[["model_build_label", "mode", "feature_family_name", "feature_policy", "feature_transform_label", SCORE_COL, "mae", "rmse", "r2"]])

path = forecast_paths[forecast_paths["config_id"].eq(best_config_id)].copy()
path["target_date"] = pd.to_datetime(path["target_date"])
path["error"] = path["prediction"] - path["actual"]
path["abs_error"] = path["error"].abs()

large_error_cols = ["as_of_date", "target_date", "evaluation_period", "actual", "prediction", "error", "abs_error"]
display(path.sort_values("abs_error", ascending=False).loc[:, large_error_cols].head(20))

fig, ax = plt.subplots(figsize=(12, 4))
ax.axhline(0, color="#2f323a", linewidth=1)
ax.plot(path["target_date"], path["error"], color="#007f68", linewidth=1.5)
ax.scatter(path["target_date"], path["error"], s=12, color="#007f68")
ax.set_title("Residuals Over Time For Best Global Configuration")
ax.set_ylabel("Prediction - actual UPT")
ax.set_xlabel("Target month")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 8. Candidate Dashboard Takeaways To Promote Later

Use this notebook to collect evidence before promoting any findings into the dashboard. Good candidates should be stable across reasonable metric choices and should survive period-specific checks.

Potential next cells:

- compare top models under MAE-first vs RMSE-first ranking
- inspect whether transform wins are concentrated in one feature family or broad across families
- compare full-error distributions, not only mean errors
- identify target months that many model classes miss together
- test whether recent-period winners are meaningfully different from pre-COVID winners
- produce dashboard-ready summary tables and compact visuals for the new Insights tab
